In [0]:
from pyspark.sql.functions import *

silver_orders = spark.table("workspace.silver.silver_orders")
silver_customers = spark.table("workspace.silver.silver_customers")
silver_products = spark.table("workspace.silver.silver_products")
silver_order_items = spark.table("workspace.silver.silver_order_items")
silver_exchange_rates = spark.table("workspace.silver.silver_exchange_rates")

dim_customers = spark.table("workspace.gold.dim_customers")
dim_products = spark.table("workspace.gold.dim_products")
dim_status = spark.table("workspace.gold.dim_status")
dim_date = spark.table("workspace.gold.dim_date")

In [0]:
# FACT TABLE with currency conversion
fact_orders = silver_order_items \
    .join(silver_orders, "order_id", "left") \
    .join(dim_products, "product_id", "left") \
    .join(dim_customers, "customer_id", "left") \
    .join(silver_exchange_rates,(silver_exchange_rates.currency_code == dim_products.currency),"left") \
    .select(
        col("order_id"),
        col("customer_id"),
        col("product_id"),
        col("order_date"),
        col("standard_status").alias("status"),
        silver_orders.country_code.alias("country"),
        silver_orders.channel,
        col("quantity"),
        col("unit_price").alias("local_price"),
        round((col("unit_price") * col("exchange_rate_to_usd")),2).alias("price_usd"),
        round((col("quantity") * col("unit_price") * col("exchange_rate_to_usd")),2).alias("revenue_usd")
    )

display(fact_orders)

In [0]:
fact_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_orders")